# Bulk RNA-seq TME Deconvolution Template

用于 bulk RNA-seq 免疫/基质评分、细胞比例估计及探索性组间比较。输入 TPM、log2(TPM+1)，或带匹配特征长度的 raw counts；VST/rlog 不能转换为 TPM。

先按原物种的计数与长度计算 TPM，再转换为本物种 symbol。Native CIBERSORT 保留 MGI（鼠25类）或 HGNC（人LM22）；只有人源参考方法使用小鼠→人同源转换。ssGSEA 使用28个人源 signature，输出富集分数而非细胞比例。生产分析优先使用 templates/TME/run_analysis.R。


## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
# Raw integer counts are the default and recommended source. They are converted
# to TPM using gene lengths before deconvolution.
INPUT_MODE <- "raw_counts"       # "raw_counts" (recommended) or "expression"
RAW_COUNTS_FILE <- "./0-Data/featureCounts_merged_count.annot.tsv"
RAW_COUNTS_FORMAT <- "tsv"       # "tsv", "csv", or "excel"
EXPR_FILE <- "./0-Data/TPM_matrix.csv"
EXPR_UNIT <- "tpm"               # expression mode: "tpm" or "log2_tpm"; VST is invalid for deconvolution

# Metadata file exported by RNAseq_General; must contain sample and group columns.
META_FILE <- "./1-DEG/colData.csv"

# Gene length information for counts-to-TPM conversion. Used only when
# RAW_COUNTS_FILE is provided. Provide either GENE_LENGTH_COLUMN (in bp or kb
# controlled by GENE_LENGTH_UNIT) or GENE_START_COL + GENE_END_COL.
GENE_COLUMN <- "gene_id"          # stable unique ID preferred for TPM; Ensembl IDs are converted later
GENE_LENGTH_COLUMN <- NULL
GENE_LENGTH_UNIT <- "bp"          # "bp" or "kb"
GENE_START_COL <- "gene_start"
GENE_END_COL <- "gene_end"

SAMPLE_COLUMN <- "sample"
GROUP_COLUMN <- "condition"
GROUP_LEVELS <- NULL             # NULL = infer from metadata order

# Preserve native species for CIBERSORT; human-reference branches use a separate
# HGNC matrix. Mouse orthologs use installed babelgene data by default.
SPECIES <- "human"              # "human" or "mouse"
ORTHOLOG_CACHE <- NULL          # optional legacy biomaRt cache
RUN_SSGSEA <- TRUE              # 28 human immune signatures, true ssGSEA

# Optional custom group colors. If NULL, make_group_colors(GROUP_LEVELS) is used.
# Example: GROUP_COLORS <- c("Control" = "#6F6F6F", "Treatment" = "#E07B54")
GROUP_COLORS <- NULL

# ESTIMATE (native implementation). IOBR's estimate method is a wrapper around
# the same estimate package, so native and IOBR estimate are essentially
# identical. Keep RUN_ESTIMATE = TRUE when you want the scores independently.
RUN_ESTIMATE <- TRUE

# IOBR multi-algorithm deconvolution
RUN_IOBR <- FALSE               # enable after checking IOBR reference caches
IOBR_METHODS <- c("estimate", "cibersort", "epic", "xcell")
IOBR_PERM <- 1000
IOBR_ARRAYS <- FALSE

# Native CIBERSORT (optional, bundled in references/CIBERSORT/)
RUN_CIBERSORT <- FALSE
CIBERSORT_SCRIPT <- NULL        # NULL = auto-locate references/CIBERSORT/CIBERSORT.R
CIBERSORT_SIGNATURE <- NULL     # NULL = auto LM22 (human) or cibersort_mouse_22.csv (mouse)
CIBERSORT_PERM <- 1000
CIBERSORT_QN <- FALSE              # RNA-seq: FALSE; microarray: TRUE

# Compare only verified identical human references and QN settings.
# Mouse 25-class and human LM22 outputs are not equivalent.
RUN_CIBERSORT_COMPARISON <- TRUE

# Output
OUTDIR <- "RNAseq_TME_Deconvolution_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)

## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("tidyverse", "pheatmap", "ggpubr", "corrplot", "RColorBrewer"))
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("GSVA", "limma", "org.Mm.eg.db", "org.Hs.eg.db"))
# install.packages("estimate", repos = "http://r-forge.r-project.org")
# remotes::install_github("IOBR/IOBR")  # for IOBR-based deconvolution

suppressPackageStartupMessages({
  library(tidyverse)
  library(pheatmap)
  library(ggpubr)
  library(corrplot)
  library(limma)
})

# Project root used to locate bundled resources (e.g. references/CIBERSORT/)
repo_root <- tryCatch(rprojroot::find_root(rprojroot::is_git_root), error = function(e) getwd())

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
source(file.path(LIB_DIR, "tme_utils.R"))
source(file.path(LIB_DIR, "io_utils.R"))
source(file.path(LIB_DIR, "data_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")
requested_tme_methods <- c(RUN_ESTIMATE, RUN_IOBR, RUN_CIBERSORT, RUN_SSGSEA)
# Resolve optional packages before deciding whether human orthologs are needed.
if (RUN_ESTIMATE && !requireNamespace("estimate", quietly = TRUE)) RUN_ESTIMATE <- FALSE
if (RUN_IOBR && !requireNamespace("IOBR", quietly = TRUE)) RUN_IOBR <- FALSE
if (RUN_SSGSEA && !requireNamespace("GSVA", quietly = TRUE)) RUN_SSGSEA <- FALSE


## 3. Load Expression and Metadata

In [ ]:
meta <- read_metadata(
  META_FILE,
  sample_column = SAMPLE_COLUMN,
  required_columns = GROUP_COLUMN,
  group_column = GROUP_COLUMN,
  group_levels = GROUP_LEVELS
)
if (is.null(GROUP_LEVELS)) GROUP_LEVELS <- levels(meta[[GROUP_COLUMN]])

# Build a method-valid expression matrix. Raw counts are converted to TPM;
# expression mode only accepts TPM or log2(TPM+1), never VST/rlog.
if (!INPUT_MODE %in% c("raw_counts", "expression")) stop("INPUT_MODE must be 'raw_counts' or 'expression'.")
if (INPUT_MODE == "expression" && !EXPR_UNIT %in% c("tpm", "log2_tpm")) {
  stop("TME expression input requires 'tpm' or 'log2_tpm'; VST/rlog is not valid for deconvolution.")
}
if (INPUT_MODE == "raw_counts") {
  if (!file.exists(RAW_COUNTS_FILE)) stop("RAW_COUNTS_FILE not found: ", RAW_COUNTS_FILE)
  message("Computing TPM from raw integer counts for TME deconvolution.")

  raw_annot <- read_count_table(RAW_COUNTS_FILE, input_format = RAW_COUNTS_FORMAT)
  if (is.null(GENE_COLUMN)) GENE_COLUMN <- colnames(raw_annot)[1]
  if (!GENE_COLUMN %in% colnames(raw_annot)) stop("GENE_COLUMN not found: ", GENE_COLUMN)
  count_col_names <- intersect(colnames(raw_annot), meta[[SAMPLE_COLUMN]])
  if (!length(count_col_names)) stop("No common samples between raw counts and metadata.")

  # TPM denominator includes every counted feature before ID/symbol filtering.
  expr_tpm <- build_tme_tpm(
    raw_annot, gene_column = GENE_COLUMN, count_columns = count_col_names,
    length_column = GENE_LENGTH_COLUMN,
    start_column = GENE_START_COL, end_column = GENE_END_COL,
    length_unit = GENE_LENGTH_UNIT
  )

  # Subset to common samples
  common_samples <- intersect(colnames(expr_tpm), meta[[SAMPLE_COLUMN]])
  if (length(common_samples) == 0) {
    stop("No common samples between RAW_COUNTS_FILE and META_FILE.")
  }
  expr_tpm <- expr_tpm[, common_samples, drop = FALSE]
  meta <- meta[match(common_samples, meta[[SAMPLE_COLUMN]]), ]

  validate_expression_contract(expr_tpm, expected = "tpm")
  expr_tme_input <- as.data.frame(expr_tpm, check.names = FALSE)
  expr_is_log2_tpm <- FALSE

  # Save TPM matrix for reference
  write.csv(expr_tpm, file.path(OUTDIR, "TPM_matrix.csv"))
  cat("TPM matrix saved to", file.path(OUTDIR, "TPM_matrix.csv"), "\n")
} else {
  if (!file.exists(EXPR_FILE)) stop("EXPR_FILE not found: ", EXPR_FILE)
  expr_mat <- read_expression_matrix(EXPR_FILE, gene_column = GENE_COLUMN)

  common_samples <- intersect(colnames(expr_mat), meta[[SAMPLE_COLUMN]])
  if (length(common_samples) == 0) {
    stop("No common samples between EXPR_FILE and META_FILE.")
  }
  expr_mat <- expr_mat[, common_samples, drop = FALSE]
  meta <- meta[match(common_samples, meta[[SAMPLE_COLUMN]]), ]

  validate_expression_contract(expr_mat, expected = EXPR_UNIT)
  expr_is_log2_tpm <- identical(EXPR_UNIT, "log2_tpm")
  expr_tme_input <- as.data.frame(expr_mat, check.names = FALSE)
}

validate_samples_match(colnames(expr_tme_input), meta[[SAMPLE_COLUMN]], strict_order = TRUE)
cat("Expression:", nrow(expr_tme_input), "genes x", ncol(expr_tme_input), "samples\n")
print(table(meta[[GROUP_COLUMN]], useNA = "ifany"))

# Keep native-species symbols for native CIBERSORT; build human orthologs only
# when a human-reference method was requested and its package is available.
tme_inputs <- prepare_tme_inputs(expr_tme_input, species = SPECIES,
  is_log = expr_is_log2_tpm, need_human = RUN_ESTIMATE || RUN_IOBR || RUN_SSGSEA,
  ortholog_cache = ORTHOLOG_CACHE, outdir = OUTDIR,
  native_symbols = if (INPUT_MODE == "raw_counts" && "gene_name" %in% names(raw_annot))
    setNames(as.character(raw_annot$gene_name), as.character(raw_annot[[GENE_COLUMN]])) else NULL)
expr_native <- tme_inputs$native
expr_tme <- tme_inputs$human
cat("Native TME matrix:", nrow(expr_native), "genes x", ncol(expr_native), "samples\n")

# Resolve group colors for consistent plotting
if (is.null(GROUP_COLORS) || !all(GROUP_LEVELS %in% names(GROUP_COLORS))) {
  group_colors <- make_group_colors(GROUP_LEVELS)
} else {
  group_colors <- GROUP_COLORS[GROUP_LEVELS]
}

# Define plotting metadata before any method uses it.
group_df_for_plot <- meta[, c(SAMPLE_COLUMN, GROUP_COLUMN), drop = FALSE]


## 4. ESTIMATE Score (native implementation)

In [ ]:
if (RUN_ESTIMATE) {
  if (!requireNamespace("estimate", quietly = TRUE)) {
    message("Package 'estimate' is not installed; skipping ESTIMATE.")
  } else {
    # ESTIMATE relies on the package lazy-data `common_genes`; requireNamespace()
    # alone does not resolve it, so load it explicitly before calling the API.
    utils::data("common_genes", package = "estimate", envir = environment())
    utils::data("SI_geneset", package = "estimate", envir = environment())
    estimate_df <- data.frame(NAME = rownames(expr_tme), Description = NA, expr_tme, check.names = FALSE)
    write.table(estimate_df, file.path(OUTDIR, "estimate_input.gct"), sep = "\t", quote = FALSE, row.names = FALSE)
    estimate::filterCommonGenes(input.f = file.path(OUTDIR, "estimate_input.gct"),
                                output.f = file.path(OUTDIR, "estimate_common_genes.gct"), id = "GeneSymbol")
    estimate::estimateScore(file.path(OUTDIR, "estimate_common_genes.gct"),
                            file.path(OUTDIR, "estimate_scores.gct"), platform = "illumina")
    estimate_scores <- read.table(file.path(OUTDIR, "estimate_scores.gct"), skip = 2, header = TRUE, sep = "\t", check.names = FALSE)
    rownames(estimate_scores) <- estimate_scores$NAME
    estimate_scores <- as.data.frame(t(estimate_scores[, -(1:2)]))
    estimate_scores[[SAMPLE_COLUMN]] <- rownames(estimate_scores)
    write.csv(estimate_scores, file.path(OUTDIR, "ESTIMATE_scores.csv"), row.names = FALSE)
    cat("ESTIMATE scores saved to", file.path(OUTDIR, "ESTIMATE_scores.csv"), "\n")
  }
}


## 5. IOBR Multi-algorithm TME Deconvolution

In [ ]:
iobr_results <- list()
if (RUN_IOBR) {
  if (!requireNamespace("IOBR", quietly = TRUE)) {
    message("Package 'IOBR' is not installed; skipping IOBR deconvolution.")
  } else {
    iobr_results <- run_iobr_deconvolution(
      expr_tme,
      methods = IOBR_METHODS,
      perm = IOBR_PERM,
      arrays = IOBR_ARRAYS,
      id_column = SAMPLE_COLUMN
    )
    # Save individual results
    for (method in names(iobr_results)) {
      write.csv(iobr_results[[method]], file.path(OUTDIR, paste0("IOBR_", method, ".csv")), row.names = FALSE)
    }
    # Combine all results
    if (length(iobr_results) >= 2) {
      tme_combined <- combine_tme_results(iobr_results, id_column = SAMPLE_COLUMN)
      write.csv(tme_combined, file.path(OUTDIR, "IOBR_TME_combined.csv"), row.names = FALSE)
      cat("Combined TME table:", nrow(tme_combined), "samples x", ncol(tme_combined), "features\n")
    }
  }
}


## 6. Native CIBERSORT (optional)

In [ ]:
if (RUN_CIBERSORT) {
  if (is.null(CIBERSORT_SCRIPT)) CIBERSORT_SCRIPT <- file.path(repo_root, "references", "CIBERSORT", "CIBERSORT.R")
  if (is.null(CIBERSORT_SIGNATURE)) {
    sig_name <- if (SPECIES == "mouse") "cibersort_mouse_22.csv" else "LM22.txt"
    CIBERSORT_SIGNATURE <- file.path(repo_root, "references", "CIBERSORT", sig_name)
  }
}
native_cibersort <- NULL
if (isTRUE(RUN_CIBERSORT)) {
  native_cibersort <- run_native_cibersort(
    expr_native,
    signature_file = CIBERSORT_SIGNATURE,
    cibersort_script = CIBERSORT_SCRIPT,
    is_log = FALSE,                        # native input is already linear
    perm = CIBERSORT_PERM,
    QN = CIBERSORT_QN,
    id_column = SAMPLE_COLUMN,
    verbose = TRUE
  )
  write.csv(native_cibersort, file.path(OUTDIR, "CIBERSORT_native_results.csv"), row.names = FALSE)
  write.csv(attr(native_cibersort, "reference_coverage"),
            file.path(OUTDIR, "CIBERSORT_native_reference_coverage.csv"), row.names = FALSE)
  cat("Native CIBERSORT results saved to", file.path(OUTDIR, "CIBERSORT_native_results.csv"), "\n")

  # Native CIBERSORT stacked barplot + boxplot + per-cell-type plots
  native_cib_long <- melt_tme_results(native_cibersort, id_column = SAMPLE_COLUMN,
                                       group_df = group_df_for_plot,
                                       sample_col = SAMPLE_COLUMN, group_col = GROUP_COLUMN)
  native_cib_long <- native_cib_long |> dplyr::filter(!grepl("P-value|Correlation|RMSE", .data$cell_type))

  cib_bar_size <- calc_tme_barplot_size(n_samples = length(unique(native_cib_long[[SAMPLE_COLUMN]])),
                                        n_celltypes = length(unique(native_cib_long$cell_type)))
  plot_tme_barplot_pdf(native_cib_long, group_col = GROUP_COLUMN, sample_col = SAMPLE_COLUMN,
                       filename = file.path(OUTDIR, "CIBERSORT_native_barplot.pdf"),
                       title = "Native CIBERSORT Cell Fractions",
                       width = cib_bar_size["width"], height = cib_bar_size["height"])

  cib_box_size <- calc_tme_boxplot_size(n_celltypes = length(unique(native_cib_long$cell_type)))
  plot_tme_boxplot_pdf(native_cib_long, group_col = GROUP_COLUMN, value_col = "fraction",
                       filename = file.path(OUTDIR, "CIBERSORT_native_boxplot.pdf"),
                       title = "Native CIBERSORT Cell Fraction by Group",
                       width = cib_box_size["width"], height = cib_box_size["height"],
                       group_colors = group_colors)

  plot_tme_per_celltype_pdf(
    native_cib_long,
    group_col = GROUP_COLUMN, value_col = "fraction",
    filename_prefix = file.path(OUTDIR, "CIBERSORT_native"),
    title_prefix = "Native CIBERSORT",
    group_colors = group_colors
  )

  # Broad-category aggregation
  native_cib_cat <- aggregate_tme_by_category(
    native_cibersort,
    id_column = SAMPLE_COLUMN,
    category_map = get_cibersort_category_map(SPECIES),
    method = "sum"
  )
  write.csv(native_cib_cat$wide, file.path(OUTDIR, "CIBERSORT_native_category_results.csv"), row.names = FALSE)

  native_cib_cat_long <- native_cib_cat$long |>
    dplyr::rename(cell_type = category, fraction = value)
  if (!is.null(group_df_for_plot)) {
    native_cib_cat_long <- native_cib_cat_long |>
      dplyr::left_join(group_df_for_plot, by = SAMPLE_COLUMN)
  }

  cib_cat_bar_size <- calc_tme_barplot_size(
    n_samples = length(unique(native_cib_cat_long[[SAMPLE_COLUMN]])),
    n_celltypes = length(unique(native_cib_cat_long$cell_type))
  )
  plot_tme_barplot_pdf(native_cib_cat_long, group_col = GROUP_COLUMN, sample_col = SAMPLE_COLUMN,
                       filename = file.path(OUTDIR, "CIBERSORT_native_category_barplot.pdf"),
                       title = "Native CIBERSORT Broad Categories",
                       width = cib_cat_bar_size["width"], height = cib_cat_bar_size["height"])

  cib_cat_box_size <- calc_tme_boxplot_size(n_celltypes = length(unique(native_cib_cat_long$cell_type)))
  plot_tme_boxplot_pdf(native_cib_cat_long, group_col = GROUP_COLUMN, value_col = "fraction",
                       filename = file.path(OUTDIR, "CIBERSORT_native_category_boxplot.pdf"),
                       title = "Native CIBERSORT Broad Categories by Group",
                       width = cib_cat_box_size["width"], height = cib_cat_box_size["height"],
                       group_colors = group_colors)

  plot_tme_per_celltype_pdf(
    native_cib_cat_long,
    group_col = GROUP_COLUMN, value_col = "fraction",
    filename_prefix = file.path(OUTDIR, "CIBERSORT_native_category"),
    title_prefix = "Native CIBERSORT Category",
    group_colors = group_colors
  )
}

# Native vs IOBR CIBERSORT comparison
compare_cibersort <- isTRUE(RUN_CIBERSORT) && isTRUE(RUN_IOBR) &&
  "cibersort" %in% names(iobr_results) && isTRUE(RUN_CIBERSORT_COMPARISON) && !is.null(native_cibersort)
if (compare_cibersort) {
  compare_cibersort <- cibersort_comparison_compatible(SPECIES, CIBERSORT_SIGNATURE,
    attr(iobr_results[["cibersort"]], "cibersort_signature"), CIBERSORT_QN, IOBR_ARRAYS,
    input_max = max(expr_tme))
  if (!compare_cibersort) message("Skipping native-vs-IOBR comparison: human species, identical references, input scale and QN settings were not verified.")
}
if (compare_cibersort) {
  # compare_native_iobr_cibersort() strips IOBR's "_CIBERSORT" column suffix and
  # otherwise normalizes cell-type names, so the raw IOBR table can be passed in.
  cmp <- compare_native_iobr_cibersort(
    native_cibersort,
    iobr_results[["cibersort"]],
    id_column = SAMPLE_COLUMN,
    method = "pearson"
  )
  write.csv(cmp$summary, file.path(OUTDIR, "CIBERSORT_native_vs_IOBR_summary.csv"), row.names = FALSE)
  write.csv(cmp$long, file.path(OUTDIR, "CIBERSORT_native_vs_IOBR_long.csv"), row.names = FALSE)
  cat("Native vs IOBR CIBERSORT summary saved to", file.path(OUTDIR, "CIBERSORT_native_vs_IOBR_summary.csv"), "\n")

  n_common_cells <- length(unique(cmp$long$cell_type))
  cmp_width <- min(20, max(10, 2.8 * ceiling(sqrt(n_common_cells))))
  cmp_height <- min(18, max(8, 2.8 * ceiling(n_common_cells / ceiling(sqrt(n_common_cells)))))

  plot_cibersort_correlation_pdf(
    cmp$long,
    filename = file.path(OUTDIR, "CIBERSORT_native_vs_IOBR_correlation.pdf"),
    title = "Native vs IOBR CIBERSORT Fractions",
    width = cmp_width, height = cmp_height
  )
  plot_cibersort_difference_pdf(
    cmp$long,
    filename = file.path(OUTDIR, "CIBERSORT_native_vs_IOBR_difference.pdf"),
    title = "Native - IOBR CIBERSORT Difference",
    width = cmp_width, height = cmp_height
  )
}


## 7. TME Visualization

In [ ]:
group_df_for_plot <- meta[, c(SAMPLE_COLUMN, GROUP_COLUMN), drop = FALSE]

if (RUN_IOBR && "estimate" %in% names(iobr_results)) {
  est_long <- melt_estimate_scores(iobr_results[["estimate"]], id_column = SAMPLE_COLUMN,
                                   group_df = group_df_for_plot,
                                   sample_col = SAMPLE_COLUMN, group_col = GROUP_COLUMN)
  plot_estimate_boxplot_pdf(est_long, group_col = GROUP_COLUMN,
    filename = file.path(OUTDIR, "IOBR_ESTIMATE_scores_boxplot.pdf"),
    title = "ESTIMATE Scores by Group", group_colors = group_colors,
    save_individual = TRUE, individual_prefix = file.path(OUTDIR, "IOBR_ESTIMATE"))
  plot_tme_heatmap_pdf(iobr_results[["estimate"]], meta, group_col = GROUP_COLUMN,
    sample_col = SAMPLE_COLUMN, group_colors = group_colors,
    filename = file.path(OUTDIR, "IOBR_ESTIMATE_heatmap.pdf"), title = "IOBR ESTIMATE Scores")
}

# Fraction-based methods: complete composition and group-comparison outputs.
for (method_name in intersect(c("cibersort", "epic"), names(iobr_results))) {
  method_label <- toupper(method_name)
  method_long <- melt_tme_results(iobr_results[[method_name]], id_column = SAMPLE_COLUMN,
    group_df = group_df_for_plot, sample_col = SAMPLE_COLUMN, group_col = GROUP_COLUMN) |>
    dplyr::filter(!grepl("P-value|Correlation|RMSE", .data$cell_type))
  bar_size <- calc_tme_barplot_size(length(unique(method_long[[SAMPLE_COLUMN]])), length(unique(method_long$cell_type)))
  box_size <- calc_tme_boxplot_size(length(unique(method_long$cell_type)))
  prefix <- file.path(OUTDIR, paste0("IOBR_", method_label))
  plot_tme_barplot_pdf(method_long, group_col = GROUP_COLUMN, sample_col = SAMPLE_COLUMN,
    filename = paste0(prefix, "_barplot.pdf"), title = paste(method_label, "Cell Fractions"),
    width = bar_size["width"], height = bar_size["height"])
  plot_tme_boxplot_pdf(method_long, group_col = GROUP_COLUMN, value_col = "fraction",
    filename = paste0(prefix, "_boxplot.pdf"), title = paste(method_label, "Cell Fractions by Group"),
    width = box_size["width"], height = box_size["height"], group_colors = group_colors)
  plot_tme_per_celltype_pdf(method_long, group_col = GROUP_COLUMN, value_col = "fraction",
    filename_prefix = prefix, title_prefix = method_label, group_colors = group_colors)
}

# Broad CIBERSORT categories use the project-maintained manual mapping.
if (RUN_IOBR && "cibersort" %in% names(iobr_results)) {
  cib_cat <- aggregate_tme_by_category(iobr_results[["cibersort"]], id_column = SAMPLE_COLUMN,
    category_map = get_cibersort_category_map(SPECIES), method = "sum")
  write.csv(cib_cat$wide, file.path(OUTDIR, "IOBR_CIBERSORT_category_results.csv"), row.names = FALSE)
}

if (RUN_IOBR && "xcell" %in% names(iobr_results)) {
  plot_tme_heatmap_pdf(iobr_results[["xcell"]], meta, group_col = GROUP_COLUMN,
    sample_col = SAMPLE_COLUMN, group_colors = group_colors,
    filename = file.path(OUTDIR, "IOBR_xCell_heatmap.pdf"), title = "xCell Scores", width = 10, height = 12)
  xcell_cat <- aggregate_tme_by_category(iobr_results[["xcell"]], id_column = SAMPLE_COLUMN,
    category_map = get_xcell_category_map(), method = "mean")
  write.csv(xcell_cat$wide, file.path(OUTDIR, "IOBR_xCell_category_results.csv"), row.names = FALSE)
  plot_tme_heatmap_pdf(xcell_cat$wide, meta, group_col = GROUP_COLUMN, sample_col = SAMPLE_COLUMN,
    group_colors = group_colors, filename = file.path(OUTDIR, "IOBR_xCell_category_heatmap.pdf"),
    title = "xCell Broad Categories", width = 8, height = 7)
}

## 8. ssGSEA Immune Signature Scoring

In [ ]:
ssgsea_scores <- NULL
gs <- list()
if (isTRUE(RUN_SSGSEA)) {
  immune_scores <- run_tme_ssgsea(expr_tme, outdir = OUTDIR)
  gs <- immune_scores$gene_sets
  ssgsea_scores <- immune_scores$scores

  if (length(gs) == 0) {
    message("No immune signature genes matched the expression matrix; skipping ssGSEA. ",
            "Check that row names are human gene symbols (or that mouse-to-human conversion succeeded).")
    RUN_SSGSEA <- FALSE
  } else {
    cat("ssGSEA immune scores saved to", file.path(OUTDIR, "ssGSEA_immune_scores.csv"), "\n")
  }
}


## 9. ssGSEA Group Comparison and Heatmap

In [ ]:
if (isTRUE(RUN_SSGSEA) && !is.null(ssgsea_scores)) {
  score_df <- as.data.frame(t(ssgsea_scores)) %>% rownames_to_column(SAMPLE_COLUMN) %>% left_join(meta, by = SAMPLE_COLUMN)
  score_long <- score_df %>% pivot_longer(cols = names(gs), names_to = "signature", values_to = "score")

  score_long[[GROUP_COLUMN]] <- factor(score_long[[GROUP_COLUMN]], levels = GROUP_LEVELS)
  score_comparisons <- if (length(GROUP_LEVELS) >= 2) combn(GROUP_LEVELS, 2, simplify = FALSE) else list()
  p_box <- plot_group_boxplot_pdf(
    score_long,
    value_col = "score",
    group_col = GROUP_COLUMN,
    facet_col = "signature",
    comparisons = score_comparisons,
    method = "t.test",
    title = "ssGSEA Immune Signature Scores",
    ylab = "ssGSEA score",
    group_colors = group_colors,
    filename = file.path(OUTDIR, "ssGSEA_group_boxplot.pdf"),
    width = 12,
    height = 8
  )
  print(p_box)

  ann <- data.frame(
    Group = as.character(meta[[GROUP_COLUMN]])
  )
  rownames(ann) <- meta[[SAMPLE_COLUMN]]
  ann$Group <- factor(ann$Group, levels = GROUP_LEVELS)
  annotation_colors <- list(Group = group_colors)

  # Order samples by group, then cluster within each group, so replicates of the
  # same condition appear together while preserving within-group structure.
  order_grouped_columns_ssgsea <- function(m, group_vec, group_levels = NULL) {
    if (is.null(group_levels)) group_levels <- unique(group_vec)
    ordered_cols <- character(0)
    for (g in group_levels) {
      idx <- which(group_vec == g)
      if (length(idx) == 0) next
      if (length(idx) == 1) {
        ordered_cols <- c(ordered_cols, colnames(m)[idx])
      } else {
        sub_m <- m[, idx, drop = FALSE]
        sd_rows <- apply(sub_m, 1, stats::sd, na.rm = TRUE)
        sub_m_var <- sub_m[sd_rows > 0 | is.na(sd_rows), , drop = FALSE]
        if (ncol(sub_m_var) >= 2 && nrow(sub_m_var) >= 2) {
          d <- stats::dist(t(sub_m_var))
          hc <- stats::hclust(d)
          ordered_cols <- c(ordered_cols, colnames(sub_m)[hc$order])
        } else {
          ordered_cols <- c(ordered_cols, colnames(sub_m))
        }
      }
    }
    ordered_cols
  }

  ssgsea_col_order <- order_grouped_columns_ssgsea(ssgsea_scores, ann$Group, GROUP_LEVELS)
  ssgsea_scores_plot <- ssgsea_scores[, ssgsea_col_order, drop = FALSE]
  ann_plot <- ann[ssgsea_col_order, , drop = FALSE]

  pheatmap(ssgsea_scores_plot, annotation_col = ann_plot, annotation_colors = annotation_colors,
           scale = "row", cluster_cols = FALSE, cluster_rows = TRUE,
           filename = file.path(OUTDIR, "ssGSEA_heatmap.pdf"), width = 8, height = 7)
}

if (any(requested_tme_methods) && !isTRUE(RUN_ESTIMATE) && !length(iobr_results) &&
    is.null(native_cibersort) && is.null(ssgsea_scores)) {
  stop("None of the requested TME methods produced results. Review missing dependencies and signature coverage.")
}
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
